<a href="https://colab.research.google.com/github/Ehsan-Roohi/DSMC_Python/blob/main/Relaxation_DSMC_TAS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
# --- DSMC CODE - V25.0 - Time-Averaged Adaptive Sub-cells (Corrected Methodology) ---
# This version uses a proper time-averaged quantity to set the adaptive sub-cell configuration,
# which is then held constant for a fixed number of timesteps (ADAPTATION_INTERVAL).

import numpy as np
import matplotlib.pyplot as plt
import numba
import time
from scipy.signal import savgol_filter
import pandas as pd

# ===================================================================
# ۱. بخش شبیه‌سازی (توابع این بخش بدون تغییر هستند)
# ===================================================================
MASS_AR = 39.948e-3 / 6.022e23; KB = 1.380649e-23

@numba.jit(nopython=True)
def calculate_vhs_cross_section_numba(vr_mag):
    # ... (بدون تغییر)
    d_ref = 4.17e-10; t_ref = 273.0; omega_vhs = 0.81
    if vr_mag < 1e-9: return 1e-30
    exponent = omega_vhs - 0.5
    c_ref_sq = 2 * KB * t_ref / MASS_AR
    gamma_val = 1.04533
    d_sq = (d_ref**2) * ((c_ref_sq / vr_mag**2)**exponent) * (1 / gamma_val)
    return np.pi * d_sq

@numba.jit(nopython=True)
def perform_collisions_sbt_subcell(particles, indices_in_cell, cell_start_x, cell_width, num_sub_cells, cell_vol, dt, fnum):
    # ... (بدون تغییر)
    num_particles_in_main_cell = len(indices_in_cell)
    if num_particles_in_main_cell < 2: return
    # Clamp num_sub_cells to a minimum of 1
    num_sub_cells = max(1, num_sub_cells)
    sub_cell_width = cell_width / num_sub_cells

    sub_cell_groups = [[np.int64(x) for x in range(0)] for _ in range(num_sub_cells)]
    for p_idx in indices_in_cell:
        relative_pos = particles[p_idx, 0] - cell_start_x
        sub_cell_idx = int(relative_pos / sub_cell_width)
        if 0 <= sub_cell_idx < num_sub_cells: sub_cell_groups[sub_cell_idx].append(p_idx)
    sub_cell_vol = cell_vol / num_sub_cells
    if sub_cell_vol < 1e-30: return
    for sc_idx in range(num_sub_cells):
        indices_in_sub_cell = sub_cell_groups[sc_idx]
        num_particles_in_sub_cell = len(indices_in_sub_cell)
        if num_particles_in_sub_cell < 2: continue
        for i in range(num_particles_in_sub_cell - 1):
            p1_idx = indices_in_sub_cell[i]
            remaining_particles_count = num_particles_in_sub_cell - (i + 1)
            j_offset = np.random.randint(0, remaining_particles_count)
            j = i + 1 + j_offset; p2_idx = indices_in_sub_cell[j]
            vr = particles[p1_idx, 1:4] - particles[p2_idx, 1:4]
            vr_mag = np.sqrt(np.sum(vr**2))
            if vr_mag < 1e-9: continue
            sigma_t = calculate_vhs_cross_section_numba(vr_mag)
            weighting_factor = float(remaining_particles_count)
            collision_prob = (weighting_factor * fnum * dt * sigma_t * vr_mag) / sub_cell_vol
            if np.random.rand() < collision_prob:
                vcm = 0.5 * (particles[p1_idx, 1:4] + particles[p2_idx, 1:4])
                cos_chi = 2 * np.random.rand() - 1.0; sin_chi = np.sqrt(1.0 - cos_chi**2); phi_chi = 2.0 * np.pi * np.random.rand()
                vr_prime = np.array([vr_mag * sin_chi * np.cos(phi_chi), vr_mag * sin_chi * np.sin(phi_chi), vr_mag * cos_chi])
                particles[p1_idx, 1:4] = vcm + 0.5 * vr_prime
                particles[p2_idx, 1:4] = vcm - 0.5 * vr_prime

# --- تابع run_dsmc_simulation با ساختار جدید ---
def run_dsmc_simulation(sim_params):
    LX = sim_params['LX']; RHO_INIT = sim_params['RHO_INIT']; T_INIT = sim_params['T_INIT']
    NUM_CELLS_X = sim_params['NUM_CELLS_X']; PARTICLES_PER_CELL_INIT = sim_params['PARTICLES_PER_CELL_INIT']
    TOTAL_TIME = sim_params['TOTAL_TIME']; DT = sim_params['DT']
    MIN_PARTICLES_FOR_STATS = 2
    MAX_SUB_CELLS = sim_params['MAX_SUB_CELLS']
    ADAPTATION_INTERVAL = sim_params['ADAPTATION_INTERVAL']

    particles = initialize_shock_tube(sim_params)

    N_DENSITY_REAL = RHO_INIT / MASS_AR
    CELL_VOLUME_CONCEPTUAL = (LX / NUM_CELLS_X) * ((LX/10) * (LX/10))
    FNUM = (N_DENSITY_REAL * CELL_VOLUME_CONCEPTUAL) / PARTICLES_PER_CELL_INIT
    NUM_STEPS = int(TOTAL_TIME / DT)

    results_history = {}; cell_width = LX / NUM_CELLS_X
    results_history[0.0] = sample_properties(particles, NUM_CELLS_X, cell_width, FNUM, CELL_VOLUME_CONCEPTUAL, MIN_PARTICLES_FOR_STATS)

    subcell_count_map = np.zeros((NUM_STEPS + 1, NUM_CELLS_X))

    # Initialize subcell configuration
    subcell_config = np.ones(NUM_CELLS_X, dtype=np.int32)

    current_step = 0
    while current_step < NUM_STEPS:
        # --- PHASE A: SAMPLING FOR ADAPTATION ---
        # Accumulate cell counts over the adaptation interval
        sum_cell_counts = np.zeros(NUM_CELLS_X, dtype=np.int64)

        steps_in_block = min(ADAPTATION_INTERVAL, NUM_STEPS - current_step)

        # --- PHASE D: EXECUTION (Inner Loop) ---
        # Run for a block of steps using the *fixed* subcell_config
        for _ in range(steps_in_block):
            current_step += 1

            particles[:, 0] += particles[:, 1] * DT
            hit_left = particles[:, 0] < 0; particles[hit_left, 1] *= -1; particles[hit_left, 0] *= -1
            hit_right = particles[:, 0] > LX; particles[hit_right, 1] *= -1; particles[hit_right, 0] = 2 * LX - particles[hit_right, 0]

            cell_indices = (particles[:, 0] / cell_width).astype(np.int64)
            cell_indices = np.clip(cell_indices, 0, NUM_CELLS_X - 1)
            sorted_particle_indices = np.argsort(cell_indices)
            cell_counts = np.bincount(cell_indices, minlength=NUM_CELLS_X)

            # Accumulate for averaging
            sum_cell_counts += cell_counts

            cell_start_indices = np.concatenate((np.array([0], dtype=np.int64), np.cumsum(cell_counts[:-1])))

            for i in range(NUM_CELLS_X):
                if cell_counts[i] > 1:
                    start = cell_start_indices[i]; end = start + cell_counts[i]
                    indices_in_cell_i = sorted_particle_indices[start:end]
                    cell_start_x = i * cell_width
                    # Use the fixed subcell_config for this cell
                    perform_collisions_sbt_subcell(particles, indices_in_cell_i, cell_start_x, cell_width, subcell_config[i], CELL_VOLUME_CONCEPTUAL, DT, FNUM)

            # Store the subcell configuration used in this step
            subcell_count_map[current_step, :] = subcell_config

            # Standard property sampling for plotting
            if current_step % (NUM_STEPS / 4 if NUM_STEPS > 4 else 1) == 0 or current_step == NUM_STEPS:
                 results_history[current_step * DT] = sample_properties(particles, NUM_CELLS_X, cell_width, FNUM, CELL_VOLUME_CONCEPTUAL, MIN_PARTICLES_FOR_STATS)

        # --- PHASES B & C: ADAPTATION & UPDATE ---
        # Calculate the new subcell configuration based on the time-averaged counts
        avg_cell_counts = sum_cell_counts / steps_in_block

        # We can use avg_cell_counts directly as it is proportional to density
        max_avg_count = np.max(avg_cell_counts)
        if max_avg_count > 1e-9:
            normalized_density = avg_cell_counts / max_avg_count
            adaptive_rule = 1.0 - normalized_density

            # Update the configuration for the *next* block of steps
            subcell_config = np.clip(1 + ((MAX_SUB_CELLS - 1) * adaptive_rule).astype(np.int32), 1, MAX_SUB_CELLS)

    return results_history, subcell_count_map

def initialize_shock_tube(params):
    # ... (بدون تغییر)
    num_cells = params['NUM_CELLS_X']; lx = params['LX']
    ppc_left = params['PARTICLES_PER_CELL_INIT']; t_left = params['T_INIT']
    rho_ratio = params['RHO_RATIO']
    ppc_right = int(ppc_left / rho_ratio);
    if ppc_right < 1: ppc_right = 1
    num_cells_left = num_cells // 2; num_cells_right = num_cells - num_cells_left
    total_particles = (ppc_left * num_cells_left) + (ppc_right * num_cells_right)
    particles = np.zeros((total_particles, 4))
    cell_width = lx / num_cells; v_thermal_std = np.sqrt(KB * t_left / MASS_AR)
    current_pos = 0
    for i in range(num_cells_left):
        start_idx = current_pos; end_idx = start_idx + ppc_left
        particles[start_idx:end_idx, 0] = i * cell_width + np.random.rand(ppc_left) * cell_width
        particles[start_idx:end_idx, 1:4] = np.random.normal(0, v_thermal_std, (ppc_left, 3))
        current_pos = end_idx
    for i in range(num_cells_left, num_cells):
        start_idx = current_pos; end_idx = start_idx + ppc_right
        particles[start_idx:end_idx, 0] = i * cell_width + np.random.rand(ppc_right) * cell_width
        particles[start_idx:end_idx, 1:4] = np.random.normal(0, v_thermal_std, (ppc_right, 3))
        current_pos = end_idx
    particles[:, 1:4] -= np.mean(particles[:, 1:4], axis=0)
    print(f"Initialized {total_particles} particles for Shock Tube problem.")
    return particles

def sample_properties(particles_state, num_cells, cell_width, fnum, cell_vol, min_parts):
    # ... (بدون تغییر)
    density_profile = np.full(num_cells, np.nan); velocity_profile = np.full(num_cells, np.nan); temp_profile = np.full(num_cells, np.nan)
    cell_indices = (particles_state[:, 0] / cell_width).astype(np.int64); cell_indices = np.clip(cell_indices, 0, num_cells - 1)
    sorted_indices = np.argsort(cell_indices); counts = np.bincount(cell_indices, minlength=num_cells)
    starts = np.concatenate((np.array([0], dtype=np.int64), np.cumsum(counts[:-1])))
    for i in range(num_cells):
        num_in_cell = counts[i]
        if num_in_cell > 0: density_profile[i] = num_in_cell * fnum / cell_vol
        if num_in_cell >= min_parts:
            indices_in_cell_i = sorted_indices[starts[i]:starts[i]+num_in_cell]
            cell_velocities = particles_state[indices_in_cell_i, 1:4]
            mean_vel_cell = np.mean(cell_velocities, axis=0); velocity_profile[i] = mean_vel_cell[0]
            thermal_vel_sq = np.sum((cell_velocities - mean_vel_cell)**2)
            temp_profile[i] = (MASS_AR * thermal_vel_sq) / (3 * KB * num_in_cell) if num_in_cell > 1 else 0.0
    return {'density': density_profile, 'velocity': velocity_profile, 'temperature': temp_profile}

def plot_average_subcells(avg_subcell_map, cell_centers):
    # ... (بدون تغییر)
    plt.rcParams.update({'font.size': 18, 'axes.labelsize': 20, 'axes.titlesize': 22, 'xtick.labelsize': 16, 'ytick.labelsize': 16, 'legend.fontsize': 18})
    plt.figure(figsize=(20, 10))
    plt.title('Time-Averaged Sub-cell Count per Main Cell (Time-Averaged TAS)')
    plt.plot(cell_centers, avg_subcell_map, 'o-', linewidth=2, markersize=5, label='Time-Averaged Sub-cell Count')
    plt.xlabel('Position x (m)'); plt.ylabel('Average Number of Sub-cells')
    plt.grid(True, linestyle=':'); plt.legend(); plt.tight_layout()
    plt.savefig("dsmc_time_averaged_tas_stats.eps", format='eps')
    plt.show()

# ===================================================================
# ۲. بخش اصلی اجرا کننده و رسم نمودار
# ===================================================================
if __name__ == "__main__":
    SIMULATION_PARAMS = {
        'LX': 1.0e-6, 'RHO_INIT': 1.78, 'T_INIT': 273.0,
        'NUM_CELLS_X': 4400,
        'DT': 6.744e-13,
        'PARTICLES_PER_CELL_INIT': 10,
        'TOTAL_TIME': 0.4e-9,
        'RHO_RATIO': 10.0,
        'MAX_SUB_CELLS': 20,
        # --- New parameter for the corrected methodology ---
        'ADAPTATION_INTERVAL': 50, # Update sub-cell configuration every 50 steps
    }
    NUM_ENSEMBLE_RUNS = 1000

    print(f"--- شروع اجرای نهایی با روش صحیح زیرسلول تطبیقی (مبتنی بر چگالی میانگین زمانی) ---")
    print(f"--- Parameters: NUM_CELLS_X={SIMULATION_PARAMS['NUM_CELLS_X']}, DT={SIMULATION_PARAMS['DT']:.4e}, ADAPTATION_INTERVAL={SIMULATION_PARAMS['ADAPTATION_INTERVAL']} ---")

    # ... بقیه کد اجرا بدون تغییر است ...
    all_results = []; summed_subcell_map = np.zeros((int(SIMULATION_PARAMS['TOTAL_TIME'] / SIMULATION_PARAMS['DT']) + 1, SIMULATION_PARAMS['NUM_CELLS_X']))
    start_time_total = time.time()
    for i in range(NUM_ENSEMBLE_RUNS):
        np.random.seed(int(time.time()) + i)
        single_run_history, subcell_map = run_dsmc_simulation(SIMULATION_PARAMS)
        all_results.append(single_run_history)

        if subcell_map.shape[0] == summed_subcell_map.shape[0]:
            summed_subcell_map += subcell_map
        else:
            min_steps = min(subcell_map.shape[0], summed_subcell_map.shape[0])
            summed_subcell_map[:min_steps, :] += subcell_map[:min_steps, :]

        print(f"--- اجرای {i+1}/{NUM_ENSEMBLE_RUNS} تمام شد ---")

    time_averaged_subcells_per_cell = np.mean(summed_subcell_map / NUM_ENSEMBLE_RUNS, axis=0)
    averaged_results = {}
    sample_times = sorted(all_results[0].keys())
    for t in sample_times:
        all_densities = np.array([run[t]['density'] for run in all_results if t in run])
        all_velocities = np.array([run[t]['velocity'] for run in all_results if t in run])
        all_temperatures = np.array([run[t]['temperature'] for run in all_results if t in run])
        averaged_results[t] = {
            'density': np.nanmean(all_densities, axis=0),
            'velocity': np.nanmean(all_velocities, axis=0),
            'temperature': np.nanmean(all_temperatures, axis=0),
        }

    plt.rcParams.update({'font.size': 18, 'axes.labelsize': 20, 'axes.titlesize': 22, 'xtick.labelsize': 16, 'ytick.labelsize': 16, 'legend.fontsize': 16})
    cell_width = SIMULATION_PARAMS['LX'] / SIMULATION_PARAMS['NUM_CELLS_X']
    cell_centers = (np.arange(SIMULATION_PARAMS['NUM_CELLS_X']) + 0.5) * cell_width
    fig, axes = plt.subplots(1, 3, figsize=(28, 8), sharex=True)
    fig.suptitle(f"DSMC with Time-Averaged TAS (SBT) (N_runs={NUM_ENSEMBLE_RUNS})", fontsize=24)
    plot_info = {
        'Density': {'data_key': 'density', 'ax': axes[0], 'ylabel': 'Number Density ($m^{-3}$)'},
        'Velocity': {'data_key': 'velocity', 'ax': axes[1], 'ylabel': 'Bulk Velocity (m/s)'},
        'Temperature': {'data_key': 'temperature', 'ax': axes[2], 'ylabel': 'Temperature (K)'}
    }
    for name, p_info in plot_info.items():
        ax = p_info['ax']
        for t in sample_times:
            s_data = pd.Series(averaged_results[t][p_info['data_key']])
            valid_indices = ~s_data.isna()
            label_text = f't = {t*1e9:.2f} ns'; line_style = '--' if t == 0.0 else '-'
            if np.sum(valid_indices) > 0:
                if name in ['Velocity', 'Temperature'] and np.sum(valid_indices) > 51:
                    smoothed_data = savgol_filter(s_data[valid_indices], window_length=51, polyorder=2)
                    ax.plot(cell_centers[valid_indices], smoothed_data, linestyle=line_style, label=label_text)
                else:
                    ax.plot(cell_centers[valid_indices], s_data[valid_indices], linestyle=line_style, label=label_text)
        ax.set_xlabel('Position x (m)'); ax.set_ylabel(p_info['ylabel']); ax.set_title(f"{name} Profile"); ax.grid(True, linestyle=':'); ax.legend()
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig("dsmc_final_results.eps", format='eps')
    plt.show()

    plot_average_subcells(time_averaged_subcells_per_cell, cell_centers)